# QF600 Asset Pricing — Backtesting a Portfolio Strategy & Computing Alpha

Backtest an **investor portfolio** against a **benchmark portfolio**, both rebalanced on a fixed
schedule, then decompose the result into the part explained by market exposure and the part that is not:

$$R_p - R_f \;=\; \alpha \;+\; \beta\,(R_m - R_f) \;+\; \varepsilon$$


Documentation in this [Github link](https://github.com/cyee02/MQF/tree/main/QF600/Assignment%201%20-%20Compute%20Alpha)

---
## 1. Set up

All third-party libraries are declared here and nowhere else in the notebook.

In [1]:
# Environment bootstrap: install anything missing, then switch on an interactive table viewer.
import importlib
import subprocess
import sys


def ensure_installed(package: str, module: str | None = None) -> None:
    """pip-install `package` only if `module` cannot already be imported.

    Parameters
    ----------
    package : str         name to pass to `pip install`
    module  : str | None  import name, when it differs from `package` (e.g. "lets_plot"
                          for "lets-plot"); defaults to `package`

    Returns
    -------
    None  called for its side effect: the package is importable afterwards
    """
    try:
        importlib.import_module(module or package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


ensure_installed("yfinance")
ensure_installed("lets-plot", "lets_plot")

try:
    import google.colab                                              # noqa: F401
    IN_COLAB = True
    get_ipython().run_line_magic("load_ext", "google.colab.data_table")  # paginated tables in Colab
except ImportError:
    IN_COLAB = False                                                 # local -> Data Wrangler plugin

print(f"Running in Colab: {IN_COLAB}")

Running in Colab: True


In [2]:
import datetime as dt

import numpy as np
import pandas as pd
import yfinance as yf

from lets_plot import *
LetsPlot.setup_html()

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.6f}".format)
pd.set_option("display.max_rows", 10)

---
## 2. Parameters

Everything a user would want to change lives in this one cell.

The strategy horizon is expressed as a **start / end date** rather than a raw day count: the spec
fixes it at "01 Jan 2016", and pinning the dates avoids the calendar-days vs trading-days ambiguity.
The horizon *in trading days* is derived from the data further below.

In [3]:
INVESTOR_PORTFOLIO  = {"SOXX": 0.70,          # semiconductor ETF
                       "GLD" : 0.30}          # gold ETF

BENCHMARK_PORTFOLIO = {"IVV" : 0.60,          # S&P 500 ETF
                       "AGG" : 0.40}          # US aggregate bond ETF

START_DATE       = "2016-01-01"               # ~10 years of history
END_DATE         = dt.date.today()
REBALANCE_DAYS   = 60                         # trading days between rebalances (~1 quarter)
INITIAL_CAPITAL  = 100_000                    # USD at inception
TRADING_DAYS     = 252                        # annualisation factor
RISK_FREE_TICKER = "^TNX"                     # CBOE 10-year Treasury yield index

for name, portfolio in [("Investor", INVESTOR_PORTFOLIO), ("Benchmark", BENCHMARK_PORTFOLIO)]:
    assert np.isclose(sum(portfolio.values()), 1.0), f"{name} weights must sum to 1.0"

TICKERS = list(dict.fromkeys([*INVESTOR_PORTFOLIO, *BENCHMARK_PORTFOLIO]))   # de-duplicated
TICKERS

['SOXX', 'GLD', 'IVV', 'AGG']

---
## 3. Data

### 3.1 Daily close prices

The download is kept twice: `prices_raw` exactly as returned, so §3.2 can measure what is
missing, and `prices` restricted to dates every ticker traded, which is what the backtest uses.

In [4]:
prices_raw =\
(
    yf.download(TICKERS,                # every ticker in one request
                start        = START_DATE,
                end          = END_DATE,
                auto_adjust  = True,    # adjust for splits & dividends -> total-return prices
                progress     = False
               )
    ["Close"]                           # daily close only
    [TICKERS]                           # restore the declared column order
)

prices =\
(
    prices_raw
    .dropna()                           # keep only dates every ticker traded
)

prices

Ticker,SOXX,GLD,IVV,AGG
Date,,,,
2016-01-04,26.684082,102.889999,170.074753,79.940125
2016-01-05,26.431665,103.180000,170.428375,79.977158
2016-01-06,25.518162,104.669998,168.197159,80.280746
2016-01-07,24.802975,106.150002,164.172668,80.273331
2016-01-08,24.445372,105.680000,162.353973,80.451035
...,...,...,...,...
2026-08-31,511.040009,408.420013,770.739990,97.072998
2026-09-01,500.309998,396.750000,765.349976,96.769997
2026-09-02,501.440002,402.779999,768.809998,96.839996


### 3.2 Data completeness

Measured on `prices_raw`, before `.dropna()` hid anything. A ticker can be short of days for two
very different reasons, so they are counted separately:

- **a late start or early end** — the fund simply did not exist on those dates (`SOXX` and `GLD`
  both predate 2016, but this catches it for any ticker swapped into §2);
- **interior gaps** — a day inside the ticker's own coverage where every other ticker printed a
  price and this one did not. These are the ones worth worrying about.

Any date where even one ticker is missing is dropped from the backtest, so the shared calendar is
the intersection, not the union.

In [5]:
calendar_days = len(prices_raw)          # union calendar: any date at least one ticker traded

gaps_within_coverage =\
(
    prices_raw
    .apply(lambda column: column
                          .loc[column.first_valid_index():column.last_valid_index()]
                          .isna()
                          .sum())      # missing days strictly inside each ticker's own history
)

completeness =\
(
    pd.DataFrame({"First Quote"  : prices_raw.apply(lambda column: column.first_valid_index()),
                  "Last Quote"   : prices_raw.apply(lambda column: column.last_valid_index()),
                  "Trading Days" : calendar_days,
                  "Observed"     : prices_raw.count(),
                  "Missing"      : prices_raw.isna().sum(),
                  "Interior Gaps": gaps_within_coverage})
    .assign(**{"Missing %": lambda frame: (frame["Missing"] / calendar_days).map("{:.2%}".format)})
    .rename_axis("Ticker")
)

print(f"Union calendar : {prices_raw.index[0]:%d %b %Y} -> {prices_raw.index[-1]:%d %b %Y} "
      f"({calendar_days:,} trading days, the union across all tickers)")
print(f"Shared calendar: {len(prices):,} days kept by .dropna() "
      f"({calendar_days - len(prices):,} dropped, {1 - len(prices) / calendar_days:.2%})")

if gaps_within_coverage.any():
    print("Interior gaps (a ticker missing a day the others traded): "
          + ", ".join(f"{ticker} {count}"
                      for ticker, count in gaps_within_coverage[gaps_within_coverage > 0].items()))
else:
    print("No interior gaps: every ticker quotes a price on every day inside its own coverage.")

completeness

Union calendar : 04 Jan 2016 -> 04 Sep 2026 (2,684 trading days, the union across all tickers)
Shared calendar: 2,684 days kept by .dropna() (0 dropped, 0.00%)
No interior gaps: every ticker quotes a price on every day inside its own coverage.


,First Quote,Last Quote,Trading Days,Observed,Missing,Interior Gaps,Missing %
Ticker,,,,,,,
SOXX,2016-01-04,2026-09-04,2684,2684,0,0,0.00%
GLD,2016-01-04,2026-09-04,2684,2684,0,0,0.00%
IVV,2016-01-04,2026-09-04,2684,2684,0,0,0.00%
AGG,2016-01-04,2026-09-04,2684,2684,0,0,0.00%


### 3.3 Risk-free rate

`^TNX` quotes the 10-year Treasury yield as an annualised **percent** (e.g. `2.27` = 2.27%). It is
converted to a compounded daily rate and aligned to the trading calendar of `prices`, so it can be
subtracted from daily returns directly.

In [6]:
rf_daily =\
(
    yf.download(RISK_FREE_TICKER,
                start        = START_DATE,
                end          = END_DATE,
                auto_adjust  = True,
                progress     = False
               )
    ["Close"]
    .squeeze()                          # single-column frame -> Series
    .div(100)                           # percent -> decimal, annualised
    .add(1)
    .pow(1 / TRADING_DAYS)              # annual -> daily, compounded
    .sub(1)
    .reindex(prices.index)              # onto the equity trading calendar
    .ffill()                            # carry the last quote over yield-market holidays
    .bfill()                            # cover a missing first observation
    .rename("rf_daily")
)

rf_daily.to_frame()

,rf_daily
Date,
2016-01-04,0.000088
2016-01-05,0.000088
2016-01-06,0.000085
2016-01-07,0.000085
2016-01-08,0.000084
...,...
2026-08-31,0.000184
2026-09-01,0.000186
2026-09-02,0.000186


### 3.4 Daily asset returns and the derived horizon

Day 0 is inception, so its return is forced to `0` rather than dropped — this keeps every curve
starting at exactly `INITIAL_CAPITAL` on the first date.

In [7]:
asset_returns =\
(
    prices
    .pct_change()                       # daily simple returns
    .fillna(0)                          # day 0 = inception, no return yet
)

HORIZON_DAYS = len(asset_returns)       # horizon, in trading days

print(f"Horizon: {prices.index[0]:%d %b %Y} -> {prices.index[-1]:%d %b %Y} "
      f"({HORIZON_DAYS:,} trading days, {HORIZON_DAYS / TRADING_DAYS:.2f} years)")
print(f"Number of Rebalances over the horizon: {(HORIZON_DAYS - 1) // REBALANCE_DAYS}")

asset_returns

Horizon: 04 Jan 2016 -> 04 Sep 2026 (2,684 trading days, 10.65 years)
Number of Rebalances over the horizon: 44


Ticker,SOXX,GLD,IVV,AGG
Date,,,,
2016-01-04,0.000000,0.000000,0.000000,0.000000
2016-01-05,-0.009459,0.002819,0.002079,0.000463
2016-01-06,-0.034561,0.014441,-0.013092,0.003796
2016-01-07,-0.028027,0.014140,-0.023927,-0.000092
2016-01-08,-0.014418,-0.004428,-0.011078,0.002214
...,...,...,...,...
2026-08-31,0.004758,-0.001149,-0.002924,-0.000821
2026-09-01,-0.020996,-0.028574,-0.006993,-0.003121
2026-09-02,0.002259,0.015198,0.004521,0.000723


---
## 4. Reusable functions

Four small building blocks, each doing one job: rebalance a portfolio, measure a drawdown,
summarise risk & return, and regress one excess-return series on another.

In [8]:
def do_rebalance(asset_returns: pd.DataFrame,
                 weights: dict[str, float],
                 initial_capital: float,
                 rebalance_days: int) -> pd.Series:
    """Compound a portfolio daily, snapping back to target `weights` every `rebalance_days`.

    Weights are assumed perfectly divisible: no share counts, no transaction costs.

    Between rebalances each holding drifts with its own return; on a rebalance date the
    portfolio value is redistributed across the target weights and compounding resumes.
    Vectorised by splitting the horizon into fixed-length blocks, so there is no day loop.

    Parameters
    ----------
    asset_returns   : pd.DataFrame      daily simple returns; DatetimeIndex rows x ticker
                                        columns, first row 0.0 (inception)
    weights         : dict[str, float]  target weight per ticker, summing to 1.0; the keys
                                        must be a subset of `asset_returns.columns`
    initial_capital : float             portfolio value on the first date, in dollars
    rebalance_days  : int               trading days between rebalances

    Returns
    -------
    pd.Series  float, named "Value", indexed like `asset_returns`: the portfolio's dollar
               value on each date, starting at `initial_capital`
    """
    w      = pd.Series(weights, dtype = float)
    blocks = pd.Series(np.arange(len(asset_returns)) // rebalance_days,
                       index = asset_returns.index)

    value_per_dollar =\
    (
        asset_returns
        [w.index]                        # only this portfolio's assets, in weight order
        .add(1)
        .groupby(blocks)
        .cumprod()                       # growth of $1 per asset, restarting each block
        .mul(w, axis = "columns")        # weighted at the block's target weights
        .sum(axis = "columns")           # -> portfolio value per $1 invested at block start
    )

    capital_at_block_start =\
    (
        value_per_dollar
        .groupby(blocks)
        .last()                          # each block's total growth factor
        .shift(1, fill_value = 1.0)      # block N starts with what blocks 0..N-1 earned
        .cumprod()
        .mul(initial_capital)
    )

    return (
        value_per_dollar
        .mul(blocks.map(capital_at_block_start))
        .rename("Value")
    )

In [9]:
def compute_drawdown(value: pd.Series) -> pd.Series:
    """Percentage decline from the running peak, for every date.

    Parameters
    ----------
    value : pd.Series  a positive level series, e.g. the dollar values from `do_rebalance`

    Returns
    -------
    pd.Series  float, indexed like `value`: 0.0 on a day that sets a new peak, negative
               below it (-0.25 = 25% under water)
    """
    return (
        value
        .div(value.cummax())
        .sub(1)
    )

In [10]:
def compute_metrics(portfolio_returns: pd.Series,
                    rf_daily: pd.Series,
                    trading_days: int) -> dict[str, float]:
    """Return / volatility / Sharpe, reported both over the full horizon and annualised.

    Parameters
    ----------
    portfolio_returns : pd.Series  daily simple returns of one portfolio
    rf_daily          : pd.Series  daily risk-free rate, sharing the same index
    trading_days      : int        annualisation factor (252)

    Returns
    -------
    dict[str, float]  six metrics keyed by name: "Horizon Return", "Annualised Return",
                      "Horizon Volatility", "Annualised Volatility", "Horizon Sharpe",
                      "Annualised Sharpe". Returns and volatilities are decimals
                      (0.12 = 12%); the Sharpe ratios are unitless.
    """
    horizon_days   = len(portfolio_returns)
    horizon_return = portfolio_returns.add(1).prod() - 1
    daily_vol      = portfolio_returns.std()

    excess_returns = portfolio_returns.sub(rf_daily)
    annual_sharpe  = excess_returns.mean() / excess_returns.std() * np.sqrt(trading_days)

    return {
        "Horizon Return"       : horizon_return,
        "Annualised Return"    : (1 + horizon_return) ** (trading_days / horizon_days) - 1,
        "Horizon Volatility"   : daily_vol * np.sqrt(horizon_days),
        "Annualised Volatility": daily_vol * np.sqrt(trading_days),
        "Horizon Sharpe"       : annual_sharpe * np.sqrt(horizon_days / trading_days),
        "Annualised Sharpe"    : annual_sharpe,
    }

In [11]:
def compute_alpha_beta(excess_investor: pd.Series,
                       excess_benchmark: pd.Series,
                       trading_days: int) -> dict[str, float]:
    """OLS of the investor's excess return on the benchmark's excess return.

        R_p - R_f = alpha + beta * (R_m - R_f) + epsilon

    beta  -> sensitivity to the benchmark (just the market)
    alpha -> average excess return the benchmark does not explain (skill)

    Parameters
    ----------
    excess_investor  : pd.Series  daily investor excess returns  (R_p - R_f), the regressand
    excess_benchmark : pd.Series  daily benchmark excess returns (R_m - R_f), the regressor;
                                  same index as `excess_investor`
    trading_days     : int        annualisation factor (252)

    Returns
    -------
    dict[str, float]  "Beta" (unitless slope), "Alpha (daily)" (decimal intercept),
                      "Annualised Alpha" (decimal, alpha compounded over `trading_days`),
                      "R-squared" (0.0 to 1.0)
    """
    beta        = excess_investor.cov(excess_benchmark) / excess_benchmark.var()
    alpha_daily = excess_investor.mean() - beta * excess_benchmark.mean()

    return {
        "Beta"             : beta,
        "Alpha (daily)"    : alpha_daily,
        "Annualised Alpha" : (1 + alpha_daily) ** trading_days - 1,
        "R-squared"        : excess_investor.corr(excess_benchmark) ** 2,
    }

---
## 5. Backtest

### 5.1 Portfolio values

Both portfolios start at `INITIAL_CAPITAL` and compound daily, rebalancing every
`REBALANCE_DAYS`.

In [12]:
portfolio_values =\
(
    pd.DataFrame(
        {"Investor" : do_rebalance(asset_returns, INVESTOR_PORTFOLIO,
                                   INITIAL_CAPITAL, REBALANCE_DAYS),
         "Benchmark": do_rebalance(asset_returns, BENCHMARK_PORTFOLIO,
                                   INITIAL_CAPITAL, REBALANCE_DAYS)}
    )
    .rename_axis("Date")
)

portfolio_values

,Investor,Benchmark
Date,,
2016-01-04,"100,000.000000","100,000.000000"
2016-01-05,"99,422.395428","100,143.283472"
2016-01-06,"97,460.457307","99,508.049572"
2016-01-07,"96,015.846303","98,084.554859"
2016-01-08,"94,940.710970","97,531.863091"
...,...,...
2026-08-31,"1,358,144.286115","274,185.005619"
2026-09-01,"1,326,102.132966","272,684.520089"
2026-09-02,"1,334,946.731586","273,509.603934"


### 5.2 Portfolio returns and excess returns

Rebalancing is value-neutral (it only reshuffles an unchanged total), so the daily return of the
portfolio is simply the day-over-day change in its value, including across rebalance dates.

In [13]:
portfolio_returns =\
(
    portfolio_values
    .pct_change()
    .fillna(0)                           # day 0 = inception
)

portfolio_returns

,Investor,Benchmark
Date,,
2016-01-04,0.000000,0.000000
2016-01-05,-0.005776,0.001433
2016-01-06,-0.019733,-0.006343
2016-01-07,-0.014823,-0.014305
2016-01-08,-0.011197,-0.005635
...,...,...
2026-08-31,0.002726,-0.002099
2026-09-01,-0.023593,-0.005473
2026-09-02,0.006670,0.003026


In [14]:
excess_returns =\
(
    portfolio_returns
    .sub(rf_daily, axis = "index")       # R - R_f, the input to Sharpe and to alpha/beta
)

excess_returns

,Investor,Benchmark
Date,,
2016-01-04,-0.000088,-0.000088
2016-01-05,-0.005864,0.001345
2016-01-06,-0.019819,-0.006429
2016-01-07,-0.014907,-0.014390
2016-01-08,-0.011281,-0.005718
...,...,...
2026-08-31,0.002542,-0.002283
2026-09-01,-0.023779,-0.005658
2026-09-02,0.006484,0.002840


---
## 6. Result analysis

### 6.1 Return, volatility and Sharpe

In [15]:
metrics_table =\
(
    pd.DataFrame(
        {column: compute_metrics(portfolio_returns[column], rf_daily, TRADING_DAYS)
         for column in portfolio_values.columns}
    )
    .rename_axis("Metric")
)

metrics_table

,Investor,Benchmark
Metric,,
Horizon Return,12.716730,1.747910
Annualised Return,0.278722,0.099557
Horizon Volatility,0.809404,0.356377
Annualised Volatility,0.248013,0.109199
Horizon Sharpe,3.279124,2.192056
Annualised Sharpe,1.004770,0.671677


### 6.2 Drawdowns

In [16]:
drawdowns =\
(
    portfolio_values
    .apply(compute_drawdown)
)

max_drawdown_table =\
(
    pd.DataFrame({"Max Drawdown"     : drawdowns.min(),
                  "Max Drawdown Date": drawdowns.idxmin()})
    .rename_axis("Portfolio")
)

print("Investor portfolio worst loss from a prior peak: "
      f"{drawdowns['Investor'].min():.2%} on {drawdowns['Investor'].idxmin():%d %b %Y}")

max_drawdown_table

Investor portfolio worst loss from a prior peak: -35.79% on 14 Oct 2022


,Max Drawdown,Max Drawdown Date
Portfolio,,
Investor,-0.357920,2022-10-14
Benchmark,-0.209590,2020-03-23


### 6.3 Alpha and beta

Investor excess return regressed on benchmark excess return.

In [17]:
alpha_beta = compute_alpha_beta(excess_returns["Investor"],
                                excess_returns["Benchmark"],
                                TRADING_DAYS)

print(f"R_p - R_f = {alpha_beta['Alpha (daily)']:.6f} "
      f"+ {alpha_beta['Beta']:.4f} * (R_m - R_f)      [daily, R^2 = {alpha_beta['R-squared']:.4f}]")

pd.Series(alpha_beta).rename("Investor vs Benchmark").to_frame()

R_p - R_f = 0.000480 + 1.7491 * (R_m - R_f)      [daily, R^2 = 0.5932]


,Investor vs Benchmark
Beta,1.749107
Alpha (daily),0.000480
Annualised Alpha,0.128475
R-squared,0.593167


### 6.4 Summary table

Every computed figure in one place: returns, volatility, Sharpe, max drawdown and its date,
plus alpha and beta.

In [18]:
summary_table =\
(
    metrics_table
    .T                                                       # portfolios -> rows
    .assign(**{"Max Drawdown"     : max_drawdown_table["Max Drawdown"],
               "Max Drawdown Date": max_drawdown_table["Max Drawdown Date"],
               "Beta"             : [alpha_beta["Beta"],             np.nan],
               "Annualised Alpha" : [alpha_beta["Annualised Alpha"], np.nan],
               "R-squared"        : [alpha_beta["R-squared"],        np.nan]})
    .T                                                       # metrics -> rows
    .rename_axis("Metric")
)

summary_table

,Investor,Benchmark
Metric,,
Horizon Return,12.716730,1.747910
Annualised Return,0.278722,0.099557
Horizon Volatility,0.809404,0.356377
Annualised Volatility,0.248013,0.109199
Horizon Sharpe,3.279124,2.192056
Annualised Sharpe,1.004770,0.671677
Max Drawdown,-0.357920,-0.209590
Max Drawdown Date,2022-10-14 00:00:00,2020-03-23 00:00:00
Beta,1.749107,NaN


In [19]:
# Same table, formatted for reading (alpha and beta describe the investor vs the benchmark,
# so they are blank in the benchmark's own column).
percent_metrics = ["Horizon Return", "Annualised Return", "Horizon Volatility",
                   "Annualised Volatility", "Max Drawdown", "Annualised Alpha"]

summary_display =\
(
    summary_table
    .apply(lambda row: row.map(lambda v: f"{v:.2%}"     if row.name in percent_metrics    else
                                         f"{v:%d %b %Y}" if row.name == "Max Drawdown Date" else
                                         f"{v:.4f}")
                          .where(row.notna(), "—"),
           axis = "columns")
)

summary_display

,Investor,Benchmark
Metric,,
Horizon Return,1271.67%,174.79%
Annualised Return,27.87%,9.96%
Horizon Volatility,80.94%,35.64%
Annualised Volatility,24.80%,10.92%
Horizon Sharpe,3.2791,2.1921
Annualised Sharpe,1.0048,0.6717
Max Drawdown,-35.79%,-20.96%
Max Drawdown Date,14 Oct 2022,23 Mar 2020
Beta,1.7491,—


---
## 7. Charts

In [20]:
portfolio_values_long =\
(
    portfolio_values
    .reset_index()
    .melt(id_vars    = "Date",
          var_name   = "Portfolio",
          value_name = "Value")
)

portfolio_values_long

,Date,Portfolio,Value
0,2016-01-04,Investor,"100,000.000000"
1,2016-01-05,Investor,"99,422.395428"
2,2016-01-06,Investor,"97,460.457307"
3,2016-01-07,Investor,"96,015.846303"
4,2016-01-08,Investor,"94,940.710970"
...,...,...,...
5363,2026-08-31,Benchmark,"274,185.005619"
5364,2026-09-01,Benchmark,"272,684.520089"
5365,2026-09-02,Benchmark,"273,509.603934"
5366,2026-09-03,Benchmark,"275,428.908858"


In [21]:
equity_plot =\
(
    ggplot(portfolio_values_long,
           aes(x = "Date",
               y = "Value")
          )
    + geom_line(aes(color = "Portfolio"),
                size = 0.8)
    + scale_color_manual(values = ["blue", "red"],
                         labels = ["Benchmark: IVV 60 / AGG 40",
                                   "Investor: SOXX 70 / GLD 30"],
                         name   = "Portfolio")
    + scale_y_continuous(format = "$,.0f")
    + labs(title    = f"Growth of ${INITIAL_CAPITAL:,.0f}, rebalanced every {REBALANCE_DAYS} trading days",
           subtitle = f"{prices.index[0]:%d %b %Y} to {prices.index[-1]:%d %b %Y}",
           x        = "",
           y        = "Portfolio Value")
    + ggsize(1000, 500)
    + theme(legend_position = "top")
)

equity_plot

In [22]:
drawdown_plot =\
(
    ggplot(drawdowns.reset_index()
                    .melt(id_vars    = "Date",
                          var_name   = "Portfolio",
                          value_name = "Drawdown"),
           aes(x = "Date",
               y = "Drawdown")
          )
    + geom_area(aes(fill = "Portfolio"),
                alpha = 0.35)
    + geom_line(aes(color = "Portfolio"),
                size = 0.5)
    + scale_fill_manual(values  = ["blue", "red"], name = "Portfolio")
    + scale_color_manual(values = ["blue", "red"], name = "Portfolio")
    + scale_y_continuous(format = ".0%")
    + labs(title = "Underwater plot: decline from the running peak",
           x     = "",
           y     = "Drawdown")
    + ggsize(1000, 400)
    + theme(legend_position = "top")
)

drawdown_plot

In [23]:
# The regression behind alpha and beta, drawn: each point is one trading day.
alpha_beta_plot =\
(
    ggplot(excess_returns.reset_index(),
           aes(x = "Benchmark",
               y = "Investor")
          )
    + geom_point(color = "grey",
                 alpha = 0.20,
                 size  = 1.5)
    + geom_abline(slope     = alpha_beta["Beta"],
                  intercept = alpha_beta["Alpha (daily)"],
                  color     = "red",
                  size      = 1.0)
    + geom_hline(yintercept = 0, color = "black", size = 0.3)
    + geom_vline(xintercept = 0, color = "black", size = 0.3)
    + scale_x_continuous(format = ".1%")
    + scale_y_continuous(format = ".1%")
    + labs(title    = (f"R_p - R_f  =  {alpha_beta['Alpha (daily)']:.6f}  "
                       f"+  {alpha_beta['Beta']:.4f} (R_m - R_f)"),
           subtitle = (f"Daily excess returns  |  R-squared = {alpha_beta['R-squared']:.4f}  |  "
                       f"annualised alpha = {alpha_beta['Annualised Alpha']:.2%}"),
           x        = "Benchmark excess return  (R_m - R_f)",
           y        = "Investor excess return  (R_p - R_f)")
    + ggsize(700, 700)
)

alpha_beta_plot

In [24]:
dashboard =\
(
    gggrid([equity_plot, drawdown_plot],
           ncol = 1)
    + ggsize(1000, 900)
)

dashboard

---
## 8. Sharing on Google Colab

Upload this `.ipynb` to Google Drive and open it with Colab (or use **File → Upload notebook** at
[colab.research.google.com](https://colab.research.google.com)).

Section 1 detects Colab automatically: it installs `yfinance` and `lets-plot`, and loads
`google.colab.data_table`, which renders every DataFrame below as a sortable, paginated,
filterable table — the Colab counterpart to the Data Wrangler plugin used locally. No other
change is needed; **Runtime → Run all** reproduces the whole analysis.